In [ ]:
# --- Install & Imports ---
!pip -q install holidays > /dev/null

import os, re, io, json, glob, zipfile, math, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# -----------------
# Config & Paths
# -----------------
BASE_DIR      = "/content/drive/MyDrive/aimers_final/data"
BASE2_DIR     = "/content/drive/MyDrive/aimers_final/data/train"
TRAIN_PATH    = os.path.join(BASE2_DIR, "train.csv")
SAMPLE_SUB    = os.path.join(BASE_DIR, "sample_submission.csv")
BASE_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(BASE_CKPT_DIR, exist_ok=True)

META_DIR      = os.path.join(BASE2_DIR, "meta")
PATH_WEATHER  = os.path.join(META_DIR, "TRAIN_weather.csv")
PATH_ROOM     = os.path.join(META_DIR, "TRAIN_room.csv")
PATH_GROUP    = os.path.join(META_DIR, "TRAIN_group.csv")
PATH_SKI      = os.path.join(META_DIR, "TRAIN_ski.csv")
PATH_HWADAM   = os.path.join(META_DIR, "TRAIN_hwadam.csv")
PATH_PRICE    = os.path.join(BASE_DIR, "price.csv")
PATH_RTYPE    = os.path.join(BASE_DIR, "room_type.csv")
PATH_MAP_IMG  = os.path.join(BASE_DIR, "Map.jpg")  # (선택) 구역 매핑에 사용하면 좋음

ENC_LEN = 28
DEC_LEN = 7

BATCH_SIZE = 256
EPOCHS     = 20
LR         = 1e-3
WD         = 1e-4
HIDDEN     = 128
LAYERS     = 2
DROPOUT    = 0.1
TEACHER_FORCING = 0.5
VALID_DAYS = max(ENC_LEN + DEC_LEN, 42)

USE_ATTENTION   = False
USE_DOW_ONEHOT  = True

SPECIAL_STORES   = ["담하", "미라시아"]
SPECIAL_STORE_WT = 2.0

SEEDS  = [42, 123]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Feature hyperparameters
THETA_CLOSED = 0.15
TAU_COOL     = 5.0
Z_THR_SPIKE  = 2.0
ROLW7        = 7
ROLW28       = 28

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# -----------------
# CSV normalize
# -----------------
def _clean(s) -> str:
    s = str(s).replace("\ufeff", "")
    return re.sub(r"\s+", "", s).lower()

def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    raw = list(df.columns)
    norm = {_clean(c): c for c in raw}
    date_key = next((norm[k] for k in norm if ("영업일자" in k) or ("일자" in k) or k.endswith("date")), None)
    menu_key = next((norm[k] for k in norm if ("영업장명_메뉴명" in k) or ("업장" in k and "메뉴" in k) or ("menu" in k)), None)
    qty_key  = next((norm[k] for k in norm if ("매출수량" in k) or ("수량" in k) or (k in ("qty","quantity"))), None)

    if "영업일자" in raw: date_key = "영업일자"
    if "영업장명_메뉴명" in raw: menu_key = "영업장명_메뉴명"
    if "매출수량" in raw: qty_key = "매출수량"
    if not (date_key and menu_key and qty_key):
        raise KeyError(f"필수 컬럼 누락. 원본: {raw}")

    df = df.rename(columns={date_key:"영업일자", menu_key:"영업장명_메뉴명", qty_key:"매출수량"})
    return df[["영업일자","영업장명_메뉴명","매출수량"]]

def split_store_menu(name: str):
    name = str(name)
    if "_" in name:
        p = name.find("_")
        return name[:p], name[p+1:]
    return name, "__UNK__"

# -----------------
# Domain calendar
# -----------------
try:
    import holidays
    KR_HOLIDAYS_AVAILABLE = True
except Exception:
    KR_HOLIDAYS_AVAILABLE = False

def build_holiday_flags(dates: pd.Series, years_pad:int=1) -> pd.DataFrame:
    dates = pd.to_datetime(dates)
    dow = dates.dt.weekday.values
    is_weekend = np.isin(dow, [4,5,6]).astype(np.float32)  # 금토일
    if KR_HOLIDAYS_AVAILABLE:
        years = list(range(dates.min().year - int(years_pad), dates.max().year + 1 + int(years_pad)))
        kr = holidays.KR(years=years)
        is_h = np.array([1.0 if d.date() in kr else 0.0 for d in dates], dtype=np.float32)
    else:
        is_h = np.zeros(len(dates), dtype=np.float32)
    prev_h = np.r_[0.0, is_h[:-1]]
    next_h = np.r_[is_h[1:], 0.0]
    return pd.DataFrame({
        "dow": dow.astype(np.int16),
        "is_weekend": is_weekend,
        "is_holiday_prev": prev_h,
        "is_holiday": is_h,
        "is_holiday_next": next_h
    }, index=pd.to_datetime(dates).values)

# -----------------
# Meta loaders (결측치/이상치 포함 처리)
# -----------------
def _safe_read_csv(path):
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    if "영업일자" not in df.columns:
        df = df.rename(columns={df.columns[0]:"영업일자"})
    df["영업일자"] = pd.to_datetime(df["영업일자"], errors="coerce")
    df = df.dropna(subset=["영업일자"])
    return df

def load_weather():  # 연속형 결측: 보간 → 평균
    w = _safe_read_csv(PATH_WEATHER)
    if w is None: return None
    for c in w.columns:
        if c == "영업일자": continue
        w[c] = pd.to_numeric(w[c], errors="coerce")
        if "기온" in c or "temp" in _clean(c):
            w[c] = w[c].interpolate(limit_direction="both")
            w[c] = w[c].fillna(w[c].mean())
        else:
            w[c] = w[c].fillna(0)
        # 상위 0.5% winsorize
        q = w[c].quantile(0.995)
        w[c] = w[c].clip(upper=q)
    return w

def load_room():
    r = _safe_read_csv(PATH_ROOM)
    if r is None: return None
    for c in r.columns:
        if c=="영업일자": continue
        r[c] = pd.to_numeric(r[c], errors="coerce").fillna(0)
        q = r[c].quantile(0.995); r[c] = r[c].clip(upper=q)
    return r

def _sum_by_date(path, new_col):
    g = _safe_read_csv(path)
    if g is None: return None
    num_cols = [c for c in g.columns if c!="영업일자"]
    g[new_col] = g[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
    g = g[["영업일자", new_col]]
    q = g[new_col].quantile(0.995)
    g[new_col] = g[new_col].clip(upper=q)
    return g

def load_group():  return _sum_by_date(PATH_GROUP, "group_cnt")
def load_ski():    return _sum_by_date(PATH_SKI,   "ski_cnt")
def load_hwadam(): return _sum_by_date(PATH_HWADAM,"hwadam_cnt")

def load_price_map():
    if not os.path.exists(PATH_PRICE): return {}
    p = pd.read_csv(PATH_PRICE)
    p = p.rename(columns={p.columns[0]:"영업장명_메뉴명", p.columns[1]:"avg_price"})
    return p.groupby("영업장명_메뉴명")["avg_price"].mean().to_dict()

def load_room_type_scalar():
    if not os.path.exists(PATH_RTYPE): return {}
    r = pd.read_csv(PATH_RTYPE)
    num_cols = r.select_dtypes(include=[np.number]).columns
    return {f"rtype_{c}_avg": float(r[c].mean()) for c in num_cols}

# (선택) 매장→구역 맵 (지도 기반 수동 매핑; 없으면 UNK)
ZONE_MAP = {"담하":"central","미라시아":"central"}
def map_zone(store): return ZONE_MAP.get(store, "UNK")

# -----------------
# Train dataframe + exogenous build
# -----------------
def load_train_long(path: str):
    df = pd.read_csv(path)
    df = _normalize_columns(df)
    df["영업일자"] = pd.to_datetime(df["영업일자"], errors="coerce")
    df = df.dropna(subset=["영업일자"])
    df["매출수량"] = pd.to_numeric(df["매출수량"], errors="coerce").fillna(0).clip(lower=0)
    df[["store","menu"]] = df["영업장명_메뉴명"].apply(lambda s: pd.Series(split_store_menu(s)))
    return df

def make_full_grid(df_long: pd.DataFrame) -> pd.DataFrame:
    dates = pd.date_range(df_long["영업일자"].min(), df_long["영업일자"].max(), freq="D")
    menus = sorted(df_long["영업장명_메뉴명"].unique())
    g = (pd.MultiIndex.from_product([menus, dates], names=["영업장명_메뉴명","영업일자"])
         .to_frame(index=False)
         .merge(df_long[["영업일자","영업장명_메뉴명","매출수량","store","menu"]],
                how="left", on=["영업일자","영업장명_메뉴명"]))
    g["매출수량"] = pd.to_numeric(g["매출수량"], errors="coerce").fillna(0).clip(lower=0)
    g[["store","menu"]] = g["영업장명_메뉴명"].apply(lambda s: pd.Series(split_store_menu(s)))
    g = g.sort_values(["영업장명_메뉴명","영업일자"]).reset_index(drop=True)
    return g

def build_exogenous(date_index: pd.DatetimeIndex):
    """날짜 단위 외생변수 프레임과 칼럼 이름 목록 반환"""
    ex = pd.DataFrame({"영업일자": date_index})
    for loader in [load_weather, load_room, load_group, load_ski, load_hwadam]:
        df = loader()
        if df is not None:
            ex = ex.merge(df, on="영업일자", how="left")

    # 결측 안전 채움
    for c in ex.columns:
        if c == "영업일자": continue
        ex[c] = pd.to_numeric(ex[c], errors="coerce").fillna(0)
    ex_cols = [c for c in ex.columns if c != "영업일자"]
    return ex, ex_cols

# -----------------
# Helpers (rolling & stats)
# -----------------
def _roll_mean_safe(x: np.ndarray, w: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    n = len(x); out = np.zeros(n, dtype=np.float32)
    if n == 0: return out
    c = np.cumsum(np.r_[0.0, x])
    for i in range(n):
        a = max(0, i - w + 1)
        out[i] = (c[i+1] - c[a]) / (i - a + 1)
    return out

def _std_safe(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    if x.size <= 1: return 0.0
    return float(x.std(ddof=1))

def _bayesian_rate(successes: int, trials: int, prior_p: float, prior_strength: float) -> float:
    successes = max(0, int(successes))
    trials    = max(0, int(trials))
    m = max(1e-6, float(prior_strength))
    p = float(np.clip(prior_p, 0.0, 1.0))
    alpha = p * m; beta  = (1.0 - p) * m
    return float((successes + alpha) / (trials + alpha + beta + 1e-9))

def build_seasonal_priors_train(grid: pd.DataFrame,
                                prior_store_strength: float = 30.0,
                                prior_menu_strength: float = 10.0):
    df = grid.copy()
    df["month"] = pd.to_datetime(df["영업일자"]).dt.month
    st_day = (df.groupby(["영업일자", "store"])["매출수량"].sum().reset_index())
    st_day["open"] = (st_day["매출수량"] > 0).astype(int)
    st_day["month"] = pd.to_datetime(st_day["영업일자"]).dt.month
    mn_day = df.copy(); mn_day["open"] = (mn_day["매출수량"] > 0).astype(int)
    g_store = st_day.groupby("month")["open"].mean().to_dict()
    g_menu  = mn_day.groupby("month")["open"].mean().to_dict()
    s_agg = st_day.groupby(["store", "month"])["open"].agg(["sum","count"]).reset_index().rename(columns={"sum":"succ","count":"tr"})
    store_month_prob = {}
    for _, r in s_agg.iterrows():
        p0 = g_store.get(int(r["month"]), 0.5)
        ph = _bayesian_rate(int(r["succ"]), int(r["tr"]), p0, prior_store_strength)
        store_month_prob[(str(r["store"]), int(r["month"]))] = ph
    m_agg = mn_day.groupby(["영업장명_메뉴명","store","month"])["open"].agg(["sum","count"]).reset_index().rename(columns={"sum":"succ","count":"tr"})
    menu_month_prob = {}
    for _, r in m_agg.iterrows():
        key_store = (str(r["store"]), int(r["month"]))
        p0_store  = store_month_prob.get(key_store, g_store.get(int(r["month"]), 0.5))
        ph = _bayesian_rate(int(r["succ"]), int(r["tr"]), p0_store, prior_menu_strength)
        menu_month_prob[(str(r["영업장명_메뉴명"]), int(r["month"]))] = ph
    return store_month_prob, menu_month_prob, g_store, g_menu

In [ ]:
# -----------------
# Dataset (exogenous 포함)
# -----------------
class SeqDS(Dataset):
    def __init__(self, grid: pd.DataFrame, enc_len=28, dec_len=7,
                 store2id=None, menu2id=None, holidays_df: pd.DataFrame=None,
                 add_dow_oh=True, shared_store_piv=None,
                 store_month_prob: dict = None,
                 menu_month_prob: dict = None,
                 month_global_store: dict = None,
                 month_global_menu: dict = None,
                 theta_closed: float = THETA_CLOSED,
                 tau_cool: float = TAU_COOL,
                 z_thr: float = Z_THR_SPIKE,
                 use_same_dow_mean: bool = True,
                 exog_df: pd.DataFrame = None,
                 exog_cols: list = None,
                 static_price_map: dict = None,
                 static_zone_map: dict = None):
        self.enc_len = enc_len; self.dec_len = dec_len; self.add_dow_oh = add_dow_oh
        self.theta_closed = float(theta_closed)
        self.tau_cool = float(tau_cool)
        self.z_thr = float(z_thr)
        self.use_same_dow_mean = bool(use_same_dow_mean)

        self.grid = grid.copy()
        self.dates = pd.to_datetime(sorted(self.grid["영업일자"].unique()))
        if store2id is None:
            stores = sorted(self.grid["store"].unique()); self.store2id = {s:i for i,s in enumerate(stores)}
        else:
            self.store2id = store2id
        if menu2id is None:
            menus = sorted(self.grid["영업장명_메뉴명"].unique()); self.menu2id = {m:i for i,m in enumerate(menus)}
        else:
            self.menu2id = menu2id

        key = self.grid[["영업장명_메뉴명","store"]].drop_duplicates()
        self.menu_to_store = {row["영업장명_메뉴명"]: self.store2id.get(row["store"],0) for _,row in key.iterrows()}
        self.piv_qty = (self.grid.pivot(index="영업일자", columns="영업장명_메뉴명", values="매출수량").sort_index().fillna(0.0))
        self.piv_logy = np.log1p(self.piv_qty)
        self.menu_names = list(self.piv_qty.columns)
        self.menu_ids   = np.array([self.menu2id[m] for m in self.menu_names], dtype=np.int64)
        self.menu_store_ids = np.array([self.menu_to_store[m] for m in self.menu_names], dtype=np.int64)

        if shared_store_piv is None:
            st = (self.grid.groupby(["영업일자","store"])["매출수량"].sum().reset_index())
            st["store_id"] = st["store"].map(self.store2id).astype(int)
            st_piv = (st.pivot(index="영업일자", columns="store_id", values="매출수량").sort_index().fillna(0.0))
            self.store_piv = st_piv.reindex(self.dates).fillna(0.0)
        else:
            self.store_piv = shared_store_piv.reindex(self.dates).fillna(0.0)
        self.store_open_mat = (self.store_piv.values > 0).astype(np.float32)
        self.menu_open_mat  = (self.piv_qty.values  > 0).astype(np.float32)

        if holidays_df is None:
            self.cal = build_holiday_flags(pd.Series(self.dates)); self.cal.index = self.dates
        else:
            self.cal = holidays_df.reindex(self.dates).fillna(0)

        # 외생변수 (날짜 단위)
        self.exog_cols = exog_cols or []
        if exog_df is None:
            self.exog = pd.DataFrame(0, index=self.dates, columns=self.exog_cols, dtype=np.float32)
        else:
            e = exog_df.reindex(self.dates).fillna(0)
            self.exog = e.set_index("영업일자") if ("영업일자" in e.columns) else e
            self.exog = self.exog.astype(np.float32)

        # 정적 피처(가격, 구역)
        self.price_map = static_price_map or {}
        self.zone_map  = static_zone_map  or {}

        T = len(self.dates); self.t_list = list(range(enc_len, T - dec_len + 1))
        self.n_store = len(self.store2id); self.n_menu = len(self.menu2id)
        self.store_month_prob = store_month_prob or {}
        self.menu_month_prob  = menu_month_prob  or {}
        self.month_global_store = month_global_store or {m:0.5 for m in range(1,13)}
        self.month_global_menu  = month_global_menu  or {m:0.5 for m in range(1,13)}

    def _window_month_logs(self, idx_range, store_id, menu_idx):
        enc_dates = self.dates[idx_range]
        months = np.array([d.month for d in enc_dates], dtype=np.int16)
        st_vals = self.store_piv.iloc[idx_range, store_id].to_numpy(dtype=np.float32)
        m_vals  = self.piv_qty.iloc[idx_range,  menu_idx].to_numpy(dtype=np.float32)
        out_store = np.zeros(len(idx_range), dtype=np.float32)
        out_menu  = np.zeros(len(idx_range), dtype=np.float32)
        for mth in np.unique(months):
            m_idx = (months == mth)
            out_store[m_idx] = np.log1p(st_vals[m_idx].mean()) if m_idx.any() else 0.0
            out_menu[m_idx]  = np.log1p(m_vals[m_idx].mean())  if m_idx.any() else 0.0
        return out_store, out_menu

    def _enc_returns_and_spikes(self, enc_logy: np.ndarray):
        r = np.r_[0.0, np.diff(enc_logy).astype(np.float32)]
        mu = _roll_mean_safe(r, ROLW7)
        std = np.zeros_like(r)
        for i in range(len(r)):
            a = max(0, i-ROLW7+1)
            std[i] = _std_safe(r[a:i+1])
        std = np.maximum(std, 1e-3)
        z = (r - mu) / std
        is_up = (z >  Z_THR_SPIKE).astype(np.float32)
        is_dn = (z < -Z_THR_SPIKE).astype(np.float32)
        amp_up = np.maximum(0.0, z -  Z_THR_SPIKE)
        amp_dn = np.maximum(0.0, -Z_THR_SPIKE - z)
        vol7 = np.zeros_like(r)
        for i in range(len(r)):
            a = max(0, i-ROLW7+1)
            vol7[i] = _std_safe(r[a:i+1])
        return r.astype(np.float32), z.astype(np.float32), vol7.astype(np.float32), is_up, is_dn, amp_up.astype(np.float32), amp_dn.astype(np.float32)

    def _cooldown_from_last_spike(self, z: np.ndarray):
        idxs_up = np.where(z >  Z_THR_SPIKE)[0]
        idxs_dn = np.where(z < -Z_THR_SPIKE)[0]
        last_idx = None; sign = 0; amp = 0.0
        if idxs_up.size > 0:
            last_idx = int(idxs_up[-1]); sign = +1; amp = max(0.0, z[last_idx] - Z_THR_SPIKE)
        if idxs_dn.size > 0 and (last_idx is None or idxs_dn[-1] > last_idx):
            last_idx = int(idxs_dn[-1]); sign = -1; amp = max(0.0, -Z_THR_SPIKE - z[last_idx])
        if last_idx is None:
            return np.zeros(self.dec_len, dtype=np.float32), np.zeros(self.dec_len, dtype=np.float32)
        delta = (self.enc_len - 1) - last_idx
        base = np.exp(-(max(0, delta)) / TAU_COOL) * amp
        t = np.arange(1, self.dec_len+1, dtype=np.float32)
        series = base * np.exp(-t / TAU_COOL)
        if sign > 0: return series.astype(np.float32), np.zeros_like(series, dtype=np.float32)
        else:        return np.zeros_like(series, dtype=np.float32), series.astype(np.float32)

    def _seasonal_probs_for_month(self, store_name: str, menu_name: str, month: int):
        p_store = self.store_month_prob.get((store_name, int(month)), self.month_global_store.get(int(month), 0.5))
        p_menu  = self.menu_month_prob.get((menu_name,  int(month)), self.month_global_menu.get(int(month), 0.5))
        return float(p_store), float(p_menu)

    def __len__(self): return len(self.t_list) * len(self.menu_names)

    def __getitem__(self, idx):
        t = self.t_list[idx // len(self.menu_names)]
        m = idx % len(self.menu_names)
        menu_name = self.menu_names[m]
        store_id  = self.menu_store_ids[m]
        store_name = list(self.store2id.keys())[list(self.store2id.values()).index(store_id)]

        enc_idx = list(range(t - self.enc_len, t))
        dec_idx = list(range(t, t + self.dec_len))

        enc_logy = self.piv_logy.iloc[enc_idx, m].values.astype(np.float32)
        cal_enc = self.cal.iloc[enc_idx]

        sm_log, smm_log = self._window_month_logs(enc_idx, store_id, m)
        st_open = self.store_open_mat[enc_idx, store_id].astype(np.float32)
        mn_open = self.menu_open_mat[enc_idx, m].astype(np.float32)
        st_open_r7  = _roll_mean_safe(st_open, ROLW7)
        st_open_r28 = _roll_mean_safe(st_open, ROLW28)
        mn_open_r7  = _roll_mean_safe(mn_open, ROLW7)

        r, z, vol7, is_up, is_dn, amp_up, amp_dn = self._enc_returns_and_spikes(enc_logy)

        # ---- 외생변수: Encoder 구간 실제값
        ex_enc = self.exog.iloc[enc_idx][self.exog_cols].values.astype(np.float32) if len(self.exog_cols)>0 else np.zeros((self.enc_len,0),np.float32)

        # ---- 정적 피처: 가격, 구역(one-hot)
        price = float(self.price_map.get(menu_name, 0.0))
        zone  = self.zone_map.get(store_name, "UNK")
        zone_oh = {
            "z_central":1.0 if zone=="central" else 0.0,
            "z_other":  1.0 if zone not in ("central","UNK") else 0.0,
            "z_unk":    1.0 if zone=="UNK" else 0.0
        }
        static_enc = np.column_stack([
            np.full((self.enc_len,1), price, dtype=np.float32),
            np.tile(np.array(list(zone_oh.values()), dtype=np.float32),(self.enc_len,1))
        ])

        feats = [
            enc_logy,
            cal_enc["is_weekend"].values.astype(np.float32),
            cal_enc["is_holiday_prev"].values.astype(np.float32),
            cal_enc["is_holiday"].values.astype(np.float32),
            cal_enc["is_holiday_next"].values.astype(np.float32),
            sm_log.astype(np.float32),
            smm_log.astype(np.float32),
            st_open, st_open_r7, st_open_r28,
            mn_open, mn_open_r7,
            r, z, vol7, is_up, is_dn, amp_up, amp_dn
        ]
        base_enc = np.column_stack(feats)
        x_enc = np.hstack([base_enc, ex_enc, static_enc])
        if USE_DOW_ONEHOT:
            oh = np.eye(7, dtype=np.float32)[cal_enc["dow"].values]; x_enc = np.column_stack([x_enc, oh])
        x_enc = np.nan_to_num(x_enc, copy=False)

        # ---- Decoder
        cal_dec = self.cal.iloc[dec_idx]
        dlist = [
            cal_dec["is_weekend"].values.astype(np.float32),
            cal_dec["is_holiday_prev"].values.astype(np.float32),
            cal_dec["is_holiday"].values.astype(np.float32),
            cal_dec["is_holiday_next"].values.astype(np.float32),
        ]
        months_dec = np.array([d.month for d in self.dates[dec_idx]], dtype=np.int16)
        p_store_list, p_menu_list, closed_exp, in_season = [], [], [], []
        for mo in months_dec:
            p_s, p_m = self._seasonal_probs_for_month(store_name, menu_name, int(mo))
            p_store_list.append(p_s); p_menu_list.append(p_m)
            closed = 1.0 if p_m < THETA_CLOSED else 0.0
            closed_exp.append(closed); in_season.append(1.0 - closed)
        dlist += [np.array(p_store_list, dtype=np.float32),
                  np.array(p_menu_list, dtype=np.float32),
                  np.array(closed_exp, dtype=np.float32),
                  np.array(in_season, dtype=np.float32)]
        cool_up, cool_dn = self._cooldown_from_last_spike(z)
        dlist += [cool_up.astype(np.float32), cool_dn.astype(np.float32)]

        # 같은 요일 평균
        enc_dow = self.cal.iloc[enc_idx]["dow"].values
        enc_lin = np.expm1(enc_logy)
        dow_means = {d: enc_lin[enc_dow == d].mean() if (enc_dow == d).any() else 0.0 for d in range(7)}
        same_dow = np.array([np.log1p(dow_means.get(int(d), 0.0)) for d in cal_dec["dow"].values], dtype=np.float32)
        dlist += [same_dow]

        # 외생변수: Decoder는 enc 마지막 값 carry-forward
        if len(self.exog_cols)>0:
            last_ex = ex_enc[-1]  # (E,)
            ex_dec = np.tile(last_ex, (self.dec_len, 1)).astype(np.float32)
        else:
            ex_dec = np.zeros((self.dec_len,0), np.float32)

        static_dec = np.column_stack([
            np.full((self.dec_len,1), price, dtype=np.float32),
            np.tile(np.array(list(zone_oh.values()), dtype=np.float32),(self.dec_len,1))
        ])

        base_dec = np.column_stack(dlist)
        x_dec = np.hstack([base_dec, ex_dec, static_dec])
        if USE_DOW_ONEHOT:
            dec_oh = np.eye(7, dtype=np.float32)[cal_dec["dow"].values]; x_dec = np.column_stack([x_dec, dec_oh])
        x_dec = np.nan_to_num(x_dec, copy=False)

        y_log = self.piv_logy.iloc[dec_idx, m].values.astype(np.float32)
        return {"x_enc": x_enc, "x_dec": x_dec, "y_log": y_log,
                "store_id": np.int64(store_id), "menu_id": np.int64(self.menu_ids[m]), "menu_name": menu_name}

    @staticmethod
    def collate(batch):
        x_enc = torch.tensor(np.stack([b["x_enc"] for b in batch]), dtype=torch.float32)
        x_dec = torch.tensor(np.stack([b["x_dec"] for b in batch]), dtype=torch.float32)
        y     = torch.tensor(np.stack([b["y_log"] for b in batch]), dtype=torch.float32)
        store_id = torch.tensor([b["store_id"] for b in batch], dtype=torch.long)
        menu_id  = torch.tensor([b["menu_id"] for b in batch], dtype=torch.long)
        names = [b["menu_name"] for b in batch]
        return x_enc, x_dec, y, store_id, menu_id, names

# -----------------
# Split builder (priors + exogenous)
# -----------------
def build_train_valid(valid_days=42):
    df_long = load_train_long(TRAIN_PATH)
    grid = make_full_grid(df_long)

    dates = pd.to_datetime(sorted(grid["영업일자"].unique()))
    cal_all = build_holiday_flags(pd.Series(dates)); cal_all.index = dates
    split_date = dates[-valid_days]

    # 외생변수(날짜 단위) + 정적 맵
    exog_df, exog_cols = build_exogenous(dates)
    price_map = load_price_map()
    zone_map  = {s: map_zone(s) for s in grid["store"].unique()}

    grid_train_only = grid[grid["영업일자"] < split_date].copy()
    store_month_prob, menu_month_prob, g_store, g_menu = build_seasonal_priors_train(grid_train_only)

    ds_all = SeqDS(grid, enc_len=ENC_LEN, dec_len=DEC_LEN,
                   holidays_df=cal_all, add_dow_oh=USE_DOW_ONEHOT,
                   store_month_prob=store_month_prob, menu_month_prob=menu_month_prob,
                   month_global_store=g_store, month_global_menu=g_menu,
                   exog_df=exog_df, exog_cols=exog_cols,
                   static_price_map=price_map, static_zone_map=zone_map)
    ds_tr = SeqDS(grid, enc_len=ENC_LEN, dec_len=DEC_LEN,
                  store2id=ds_all.store2id, menu2id=ds_all.menu2id,
                  holidays_df=cal_all, add_dow_oh=USE_DOW_ONEHOT,
                  store_month_prob=store_month_prob, menu_month_prob=menu_month_prob,
                  month_global_store=g_store, month_global_menu=g_menu,
                  shared_store_piv=ds_all.store_piv,
                  exog_df=exog_df, exog_cols=exog_cols,
                  static_price_map=price_map, static_zone_map=zone_map)
    ds_va = SeqDS(grid, enc_len=ENC_LEN, dec_len=DEC_LEN,
                  store2id=ds_all.store2id, menu2id=ds_all.menu2id,
                  holidays_df=cal_all, add_dow_oh=USE_DOW_ONEHOT,
                  store_month_prob=store_month_prob, menu_month_prob=menu_month_prob,
                  month_global_store=g_store, month_global_menu=g_menu,
                  shared_store_piv=ds_all.store_piv,
                  exog_df=exog_df, exog_cols=exog_cols,
                  static_price_map=price_map, static_zone_map=zone_map)

    t_all = ds_all.t_list
    ds_tr.t_list = [t for t in t_all if ds_all.dates[t] <  split_date]
    ds_va.t_list = [t for t in t_all if ds_all.dates[t] >= split_date]

    # priors + exog 메타 저장
    def _pack_priors_for_json(store_month_prob, menu_month_prob, g_store, g_menu):
        smp = {f"{k[0]}|{int(k[1])}": float(v) for k, v in store_month_prob.items()}
        mmp = {f"{k[0]}|{int(k[1])}": float(v) for k, v in menu_month_prob.items()}
        return {
            "store_month_prob": smp,
            "menu_month_prob":  mmp,
            "month_global_store": {int(k): float(v) for k, v in g_store.items()},
            "month_global_menu":  {int(k): float(v) for k, v in g_menu.items()},
        }

    priors = _pack_priors_for_json(store_month_prob, menu_month_prob, g_store, g_menu)
    meta_extra = {
        "exog_cols": exog_cols,
        "zone_keys": ["z_central","z_other","z_unk"],
        "has_price": True
    }
    return ds_tr, ds_va, cal_all, priors, meta_extra

In [ ]:
# -----------------
# Model & Loss
# -----------------
class GRUSeq2SeqMonthly(nn.Module):
    def __init__(self, n_store, n_menu, enc_feat_dim, dec_feat_dim,
                 hidden=128, layers=2, dropout=0.1, horizon=7, use_attention=False, emb_dim=32):
        super().__init__()
        self.use_attention = use_attention
        self.horizon = horizon
        self.store_emb = nn.Embedding(n_store, emb_dim)
        self.menu_emb  = nn.Embedding(n_menu,  emb_dim)
        self.enc = nn.GRU(input_size=enc_feat_dim + emb_dim*2,
                          hidden_size=hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.dec = nn.GRU(input_size=1 + dec_feat_dim + emb_dim*2,
                          hidden_size=hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x_enc, x_dec, store_id, menu_id, teacher_forcing=0.0, y_log=None):
        B, T, Fe = x_enc.size(); Hh = x_dec.size(1)
        se = self.store_emb(store_id); me = self.menu_emb(menu_id)
        e_static = torch.cat([se, me], dim=-1).unsqueeze(1).repeat(1, T, 1)
        enc_in = torch.cat([x_enc, e_static], dim=-1)
        _, h = self.enc(enc_in)
        y_prev = x_enc[:, -1:, 0:1]
        outs = []; e_static_dec = torch.cat([se, me], dim=-1).unsqueeze(1).repeat(1, Hh, 1)
        for t in range(Hh):
            x_t = torch.cat([y_prev, x_dec[:, t:t+1, :], e_static_dec[:, t:t+1, :]], dim=-1)
            o, h = self.dec(x_t, h)
            y_hat = self.head(o); outs.append(y_hat)
            if self.training and (y_log is not None) and (random.random() < teacher_forcing):
                y_prev = y_log[:, t:t+1, :].detach()
            else:
                y_prev = y_hat
        return torch.cat(outs, dim=1)

def smape_like_elem(pred, y_log):
    pred_lin = torch.expm1(pred.squeeze(-1))
    targ_lin = torch.expm1(y_log)
    denom = (pred_lin.abs() + targ_lin.abs()).clamp_min(1e-3)
    return (pred_lin - targ_lin).abs() / denom

@torch.no_grad()
def compute_metrics(pred, y_log):
    pred = pred.squeeze(-1)
    y_lin   = torch.expm1(y_log)
    pred_lin= torch.expm1(pred)
    mask = (y_lin > 0)
    if mask.sum() == 0:
        return {"SMAPE": float("nan"), "NMAE": float("nan"), "NRMSE": float("nan"), "R2": float("nan")}
    denom = (pred_lin.abs() + y_lin.abs()).clamp_min(1e-3)
    smape = ((pred_lin - y_lin).abs() / denom)[mask].mean().item()
    mae = (pred_lin - y_lin).abs()[mask].mean().item()
    rmse = torch.sqrt(((pred_lin - y_lin)[mask] ** 2).mean() + 1e-12).item()
    ybar = (y_lin[mask].mean().item() + 1e-9)
    return {"SMAPE": smape, "NMAE": mae/ybar, "NRMSE": rmse/ybar,
            "R2": (1.0 - (( (pred_lin[mask]-y_lin[mask])**2 ).sum() /
                          ((y_lin[mask]-y_lin[mask].mean())**2).sum() + 1e-12)).item()}

def detect_feat_dims(ds):
    s = ds[0]
    return int(s["x_enc"].shape[1]), int(s["x_dec"].shape[1])

# -----------------
# Train Routine
# -----------------
def train_one_seed(seed=42):
    set_seed(seed)
    print("Preparing datasets...")
    start = time.time()
    ds_tr, ds_va, cal_all, priors, meta_extra = build_train_valid(valid_days=VALID_DAYS)
    print(f"Train samples: {len(ds_tr):,} | Valid samples: {len(ds_va):,}")
    print(f"Stores: {len(ds_tr.store2id)} | Menus: {len(ds_tr.menu2id)}")

    special_ids = set([ds_tr.store2id[s] for s in SPECIAL_STORES if s in ds_tr.store2id])
    ENC_DIM, DEC_DIM = detect_feat_dims(ds_tr)
    print(f"[detect] enc_dim={ENC_DIM} | dec_dim={DEC_DIM}")

    model = GRUSeq2SeqMonthly(
        n_store=ds_tr.n_store, n_menu=ds_tr.n_menu,
        enc_feat_dim=ENC_DIM, dec_feat_dim=DEC_DIM,
        hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT,
        horizon=DEC_LEN, use_attention=USE_ATTENTION
    ).to(DEVICE)

    save_dir = os.path.join(BASE_CKPT_DIR, f"seed{seed}"); os.makedirs(save_dir, exist_ok=True)
    meta = {
        "enc_dim": ENC_DIM, "dec_dim": DEC_DIM,
        "hidden": HIDDEN, "layers": LAYERS, "dropout": DROPOUT,
        "horizon": DEC_LEN, "use_attention": USE_ATTENTION,
        "stores": list(ds_tr.store2id.keys()),
        "menus":  list(ds_tr.menu2id.keys()),
        "priors": priors,
        "theta_closed": THETA_CLOSED,
        "tau_cool": TAU_COOL,
        "z_thr": Z_THR_SPIKE,
        "use_dow_oh": USE_DOW_ONEHOT,
        "exog_cols": meta_extra["exog_cols"],
        "zone_keys": meta_extra["zone_keys"],
        "has_price": meta_extra["has_price"]
    }
    with open(os.path.join(save_dir, "model_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-5)

    tr_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  num_workers=0, collate_fn=SeqDS.collate)
    va_loader = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0, collate_fn=SeqDS.collate)

    best_val = float("inf"); best_path = os.path.join(save_dir, "best_val.pth")
    print(f"Start training on {DEVICE} | epochs={EPOCHS} | batches={len(tr_loader)}")
    for ep in range(1, EPOCHS+1):
        print(f"\n[seed {seed}] Epoch {ep:02d}/{EPOCHS}")
        model.train(); tloss_sum, tcnt = 0.0, 0
        for x_enc, x_dec, y, store_id, menu_id, _ in tqdm(tr_loader, desc="Train", dynamic_ncols=True):
            x_enc = x_enc.to(DEVICE); x_dec = x_dec.to(DEVICE)
            y_log = y.unsqueeze(-1).to(DEVICE)
            store_id = store_id.to(DEVICE); menu_id = menu_id.to(DEVICE)

            pred = model(x_enc, x_dec, store_id, menu_id, teacher_forcing=TEACHER_FORCING, y_log=y_log)
            sm = smape_like_elem(pred, y.to(DEVICE))
            y_lin = torch.expm1(y.to(DEVICE)); mask = (y_lin > 0)
            wsp = torch.ones(sm.size(0), 1, dtype=torch.float32, device=DEVICE)
            if len(special_ids) > 0:
                sp_mask = torch.isin(store_id, torch.tensor(list(special_ids), device=DEVICE)); wsp[sp_mask] = SPECIAL_STORE_WT
            loss_vec = sm * wsp; sel = loss_vec[mask]
            loss = sel.mean() if sel.numel() > 0 else loss_vec.mean()

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tloss_sum += loss.item() * x_enc.size(0); tcnt += x_enc.size(0)

        train_loss = tloss_sum / max(1, tcnt)

        model.eval(); vloss_sum, vcnt = 0.0, 0
        agg = {"SMAPE":0.0,"NMAE":0.0,"NRMSE":0.0,"R2":0.0}; mcnt=0
        for x_enc, x_dec, y, store_id, menu_id, _ in tqdm(va_loader, desc="Valid", dynamic_ncols=True):
            with torch.no_grad():
                x_enc = x_enc.to(DEVICE); x_dec = x_dec.to(DEVICE)
                pred = model(x_enc, x_dec, store_id.to(DEVICE), menu_id.to(DEVICE), teacher_forcing=0.0, y_log=None)
                sm = smape_like_elem(pred, y.to(DEVICE))
                y_lin = torch.expm1(y.to(DEVICE)); mask = (y_lin > 0)
                wsp = torch.ones(sm.size(0), 1, dtype=torch.float32, device=DEVICE)
                if len(special_ids) > 0:
                    sp_mask = torch.isin(store_id.to(DEVICE), torch.tensor(list(special_ids), device=DEVICE)); wsp[sp_mask] = SPECIAL_STORE_WT
                loss_vec = sm * wsp; sel = loss_vec[mask]
                loss = sel.mean() if sel.numel() > 0 else loss_vec.mean()
                vloss_sum += loss.item() * x_enc.size(0); vcnt += x_enc.size(0)

                metrics = compute_metrics(pred, y.to(DEVICE))
                if not any(np.isnan(list(metrics.values()))):
                    for k in agg: agg[k] += metrics[k]; mcnt += 1

        valid_loss = vloss_sum / max(1, vcnt); sched.step(valid_loss)
        if mcnt>0:
            for k in agg: agg[k] /= mcnt
            print(f"Epoch {ep:02d} | train={train_loss:.5f} | valid={valid_loss:.5f} | "
                  f"SMAPE={agg['SMAPE']:.5f} | NMAE={agg['NMAE']:.5f} | NRMSE={agg['NRMSE']:.5f} | R2={agg['R2']:.5f}")
        else:
            print(f"Epoch {ep:02d} | train={train_loss:.5f} | valid={valid_loss:.5f} | (all-zero mask)")

        if valid_loss < best_val:
            best_val = valid_loss; torch.save(model.state_dict(), best_path)
            print(f"   ↳ Saved best to {best_path} (best={best_val:.5f})")

    mins = (time.time()-start)/60
    print(f"Done. best valid = {best_val:.5f} | ckpt={best_path} | elapsed={mins:.1f}m")
    return best_val, best_path

In [ ]:
# ▶︎ Run both seeds
val, path = train_one_seed(seed=42)

Preparing datasets...
Train samples: 77,154 | Valid samples: 6,012
Stores: 8 | Menus: 167
[detect] enc_dim=58 | dec_dim=50
Start training on cpu | epochs=20 | batches=301

[seed 42] Epoch 01/20


Train:   0%|          | 0/301 [00:00<?, ?it/s]

Valid:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 01 | train=0.43951 | valid=0.39708 | SMAPE=0.06904 | NMAE=0.12860 | NRMSE=0.30069 | R2=0.15786
   ↳ Saved best to /content/drive/MyDrive/aimers_final/data/checkpoints/seed42/best_val.pth (best=0.39708)

[seed 42] Epoch 02/20


Train:   0%|          | 0/301 [00:00<?, ?it/s]

...

Epoch 04 | train=0.36070 | valid=0.37899 | SMAPE=0.06515 | NMAE=0.11826 | NRMSE=0.27781 | R2=0.17129
   ↳ Saved best to /content/drive/MyDrive/aimers_final/data/checkpoints/seed42/best_val.pth (best=0.37899)

[seed 42] Epoch 05/20


Train:   0%|          | 0/301 [00:00<?, ?it/s]

Valid:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 05 | train=0.35378 | valid=0.38242 | SMAPE=0.06556 | NMAE=0.11990 | NRMSE=0.27977 | R2=0.17047

[seed 42] Epoch 06/20


Train:   0%|          | 0/301 [00:00<?, ?it/s]

In [ ]:
# ▶︎ Run both seeds
val, path = train_one_seed(seed=1004)

In [ ]:
# --- Install & Imports ---
!pip -q install holidays > /dev/null

import os, re, io, json, glob, zipfile, math, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# -----------------
# Paths (inference)
# -----------------
BASE_CKPT_DIR = "/content/drive/MyDrive/aimers_final/data/checkpoints"
TEST_INPUT    = "/content/drive/MyDrive/aimers_final/data/test"
SAMPLE_SUB    = "/content/drive/MyDrive/aimers_final/data/sample_submission.csv"
OUT_PATH      = "/content/drive/MyDrive/aimers_final/data/final_submission.csv"

ENC_LEN = 28
DEC_LEN = 7
USE_DOW_ONEHOT = True
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS   = [42,1004]

THETA_CLOSED = 0.15
TAU_COOL     = 5.0
Z_THR_SPIKE  = 2.0
ROLW7        = 7
ROLW28       = 28

# -----------------
# Shared utils (간략화)
# -----------------
def _clean(s): return re.sub(r"\s+","",str(s)).replace("\ufeff","").lower()

def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    raw = list(df.columns); norm = {_clean(c): c for c in raw}
    d = next((norm[k] for k in norm if ("영업일자" in k) or k.endswith("date")), None)
    m = next((norm[k] for k in norm if ("영업장명_메뉴명" in k) or ("업장" in k and "메뉴" in k) or ("menu" in k)), None)
    q = next((norm[k] for k in norm if ("매출수량" in k) or ("수량" in k) or (k in ("qty","quantity"))), None)
    if "영업일자" in raw: d="영업일자"
    if "영업장명_메뉴명" in raw: m="영업장명_메뉴명"
    if "매출수량" in raw: q="매출수량"
    if not (d and m and q): raise KeyError(f"필수 컬럼 누락. 원본: {raw}")
    df = df.rename(columns={d:"영업일자", m:"영업장명_메뉴명", q:"매출수량"})
    return df[["영업일자","영업장명_메뉴명","매출수량"]]

def split_store_menu(name: str):
    if "_" in str(name):
        p = str(name).find("_"); return str(name)[:p], str(name)[p+1:]
    return str(name), "__UNK__"

try:
    import holidays
    KR_HOLIDAYS_AVAILABLE = True
except Exception:
    KR_HOLIDAYS_AVAILABLE = False

def build_holiday_flags(dates: pd.Series, years_pad:int=1) -> pd.DataFrame:
    dates = pd.to_datetime(dates); dow = dates.dt.weekday.values
    is_weekend = np.isin(dow, [4,5,6]).astype(np.float32)
    if KR_HOLIDAYS_AVAILABLE:
        years = list(range(dates.min().year - int(years_pad), dates.max().year + 1 + int(years_pad)))
        kr = holidays.KR(years=years)
        is_h = np.array([1.0 if d.date() in kr else 0.0 for d in dates], dtype=np.float32)
    else: is_h = np.zeros(len(dates), dtype=np.float32)
    prev_h = np.r_[0.0, is_h[:-1]]; next_h = np.r_[is_h[1:], 0.0]
    return pd.DataFrame({"dow": dow.astype(np.int16), "is_weekend": is_weekend,
                         "is_holiday_prev": prev_h, "is_holiday": is_h, "is_holiday_next": next_h}, index=dates.values)

def make_test_grid(df_t: pd.DataFrame):
    df = df_t.copy()
    df["영업일자"] = pd.to_datetime(df["영업일자"], errors="coerce")
    df["매출수량"] = pd.to_numeric(df["매출수량"], errors="coerce").fillna(0).clip(lower=0)
    df = df.dropna(subset=["영업일자"])
    df[["store","menu"]] = df["영업장명_메뉴명"].apply(lambda s: pd.Series(split_store_menu(s)))
    assert df["영업일자"].nunique() == ENC_LEN, f"각 TEST 파일은 정확히 {ENC_LEN}일이어야 합니다."
    last_day = df["영업일자"].max()
    future = pd.DataFrame({"영업일자": pd.date_range(last_day + pd.Timedelta(days=1), periods=DEC_LEN, freq="D")})
    menus = df["영업장명_메뉴명"].drop_duplicates().tolist()
    future = future.assign(key=1).merge(pd.DataFrame({"영업장명_메뉴명": menus, "key": [1]*len(menus)}), on="key").drop(columns=["key"])
    future["매출수량"] = 0.0
    future[["store","menu"]] = future["영업장명_메뉴명"].apply(lambda s: pd.Series(split_store_menu(s)))
    both = pd.concat([df, future], ignore_index=True)
    return both.sort_values(["영업장명_메뉴명","영업일자"]).reset_index(drop=True)

# -----------------
# Dataset & Model (decoder에서 exog=carry-forward / test에선 0 안전채움)
# -----------------
class SeqDS(Dataset):
    def __init__(self, grid: pd.DataFrame, enc_len=28, dec_len=7,
                 store2id=None, menu2id=None, holidays_df: pd.DataFrame=None,
                 add_dow_oh=True, shared_store_piv=None,
                 store_month_prob: dict = None, menu_month_prob: dict = None,
                 month_global_store: dict = None, month_global_menu: dict = None,
                 theta_closed: float = THETA_CLOSED, tau_cool: float = TAU_COOL,
                 z_thr: float = Z_THR_SPIKE, use_same_dow_mean: bool = True,
                 exog_cols=None, price_on=True, zone_keys=("z_central","z_other","z_unk")):
        self.enc_len=enc_len; self.dec_len=dec_len; self.add_dow_oh=add_dow_oh
        self.theta_closed=float(theta_closed); self.tau_cool=float(tau_cool); self.z_thr=float(z_thr)
        self.use_same_dow_mean=bool(use_same_dow_mean)
        self.grid=grid.copy(); self.dates=pd.to_datetime(sorted(self.grid["영업일자"].unique()))
        if store2id is None:
            stores=sorted(self.grid["store"].unique()); self.store2id={s:i for i,s in enumerate(stores)}
        else: self.store2id=store2id
        if menu2id is None:
            menus=sorted(self.grid["영업장명_메뉴명"].unique()); self.menu2id={m:i for i,m in enumerate(menus)}
        else: self.menu2id=menu2id
        key=self.grid[["영업장명_메뉴명","store"]].drop_duplicates()
        self.menu_to_store={row["영업장명_메뉴명"]: self.store2id.get(row["store"],0) for _,row in key.iterrows()}
        self.piv_qty=(self.grid.pivot(index="영업일자",columns="영업장명_메뉴명",values="매출수량").sort_index().fillna(0.0))
        self.piv_logy=np.log1p(self.piv_qty)
        self.menu_names=list(self.piv_qty.columns)
        self.menu_ids=np.array([self.menu2id[m] for m in self.menu_names],dtype=np.int64)
        self.menu_store_ids=np.array([self.menu_to_store[m] for m in self.menu_names],dtype=np.int64)
        if shared_store_piv is None:
            st=(self.grid.groupby(["영업일자","store"])["매출수량"].sum().reset_index())
            st["store_id"]=st["store"].map(self.store2id).astype(int)
            st_piv=(st.pivot(index="영업일자",columns="store_id",values="매출수량").sort_index().fillna(0.0))
            self.store_piv=st_piv.reindex(self.dates).fillna(0.0)
        else:
            self.store_piv=shared_store_piv.reindex(self.dates).fillna(0.0)
        self.store_open_mat=(self.store_piv.values>0).astype(np.float32)
        self.menu_open_mat =(self.piv_qty.values >0).astype(np.float32)
        if holidays_df is None:
            self.cal=build_holiday_flags(pd.Series(self.dates)); self.cal.index=self.dates
        else:
            self.cal=holidays_df.reindex(self.dates).fillna(0)
        self.t_list=list(range(enc_len, len(self.dates)-dec_len+1))
        self.n_store=len(self.store2id); self.n_menu=len(self.menu2id)
        self.store_month_prob=store_month_prob or {}; self.menu_month_prob=menu_month_prob or {}
        self.month_global_store=month_global_store or {m:0.5 for m in range(1,13)}
        self.month_global_menu =month_global_menu  or {m:0.5 for m in range(1,13)}
        # exog shape만 맞추면 되므로 test에선 0으로 채움
        self.exog_cols=list(exog_cols or [])
        self.price_on=bool(price_on)
        self.zone_keys=list(zone_keys)

    def _window_month_logs(self, idx_range, store_id, menu_idx):
        enc_dates=self.dates[idx_range]; months=np.array([d.month for d in enc_dates],dtype=np.int16)
        st_vals=self.store_piv.iloc[idx_range,store_id].to_numpy(dtype=np.float32)
        m_vals =self.piv_qty.iloc[idx_range, menu_idx].to_numpy(dtype=np.float32)
        out_store=np.zeros(len(idx_range),dtype=np.float32); out_menu=np.zeros(len(idx_range),dtype=np.float32)
        for mth in np.unique(months):
            m_idx=(months==mth);
            out_store[m_idx]=np.log1p(st_vals[m_idx].mean()) if m_idx.any() else 0.0
            out_menu[m_idx] =np.log1p(m_vals[m_idx].mean())  if m_idx.any() else 0.0
        return out_store,out_menu

    def _enc_returns_and_spikes(self, enc_logy: np.ndarray):
        r=np.r_[0.0, np.diff(enc_logy).astype(np.float32)]
        mu=_roll_mean_safe(r, ROLW7); std=np.zeros_like(r)
        for i in range(len(r)):
            a=max(0,i-ROLW7+1); std[i]=_std_safe(r[a:i+1])
        std=np.maximum(std,1e-3); z=(r-mu)/std
        is_up=(z> Z_THR_SPIKE).astype(np.float32); is_dn=(z<-Z_THR_SPIKE).astype(np.float32)
        amp_up=np.maximum(0.0, z- Z_THR_SPIKE); amp_dn=np.maximum(0.0,-Z_THR_SPIKE - z)
        vol7=np.zeros_like(r)
        for i in range(len(r)):
            a=max(0,i-ROLW7+1); vol7[i]=_std_safe(r[a:i+1])
        return r.astype(np.float32), z.astype(np.float32), vol7.astype(np.float32), is_up, is_dn, amp_up.astype(np.float32), amp_dn.astype(np.float32)

    def _seasonal_probs_for_month(self, store_name: str, menu_name: str, month: int):
        p_store=self.store_month_prob.get((store_name,int(month)), self.month_global_store.get(int(month),0.5))
        p_menu =self.menu_month_prob.get((menu_name, int(month)), self.month_global_menu.get(int(month),0.5))
        return float(p_store), float(p_menu)

    def __len__(self): return len(self.t_list) * len(self.menu_names)

    def __getitem__(self, idx):
        t=self.t_list[idx // len(self.menu_names)]; m=idx % len(self.menu_names)
        menu_name=self.menu_names[m]; store_id=self.menu_store_ids[m]
        store_name=list(self.store2id.keys())[list(self.store2id.values()).index(store_id)]
        enc_idx=list(range(t-self.enc_len, t)); dec_idx=list(range(t, t+self.dec_len))
        enc_logy=self.piv_logy.iloc[enc_idx, m].values.astype(np.float32); cal_enc=self.cal.iloc[enc_idx]
        sm_log,smm_log=self._window_month_logs(enc_idx, store_id, m)
        st_open=self.store_open_mat[enc_idx, store_id].astype(np.float32); mn_open=self.menu_open_mat[enc_idx, m].astype(np.float32)
        st_open_r7=_roll_mean_safe(st_open,ROLW7); st_open_r28=_roll_mean_safe(st_open,ROLW28); mn_open_r7=_roll_mean_safe(mn_open,ROLW7)
        r,z,vol7,is_up,is_dn,amp_up,amp_dn=self._enc_returns_and_spikes(enc_logy)

        # exog zeros (test 단계)
        ex_enc=np.zeros((self.enc_len, len(self.exog_cols)), dtype=np.float32)

        price = 0.0 if not self.price_on else 0.0
        zone_oh = {"z_central":0.0,"z_other":0.0,"z_unk":1.0}
        static_enc=np.column_stack([np.full((self.enc_len,1),price,dtype=np.float32),
                                    np.tile(np.array(list(zone_oh.values()),dtype=np.float32),(self.enc_len,1))])

        base_enc=np.column_stack([enc_logy, cal_enc["is_weekend"].values.astype(np.float32),
                                  cal_enc["is_holiday_prev"].values.astype(np.float32),
                                  cal_enc["is_holiday"].values.astype(np.float32),
                                  cal_enc["is_holiday_next"].values.astype(np.float32),
                                  sm_log.astype(np.float32), smm_log.astype(np.float32),
                                  st_open, st_open_r7, st_open_r28, mn_open, mn_open_r7,
                                  r, z, vol7, is_up, is_dn, amp_up, amp_dn])
        x_enc=np.hstack([base_enc, ex_enc, static_enc])
        if USE_DOW_ONEHOT:
            oh=np.eye(7,dtype=np.float32)[cal_enc["dow"].values]; x_enc=np.column_stack([x_enc, oh])
        x_enc=np.nan_to_num(x_enc, copy=False)

        cal_dec=self.cal.iloc[dec_idx]; dlist=[cal_dec["is_weekend"].values.astype(np.float32),
                                               cal_dec["is_holiday_prev"].values.astype(np.float32),
                                               cal_dec["is_holiday"].values.astype(np.float32),
                                               cal_dec["is_holiday_next"].values.astype(np.float32)]
        months_dec=np.array([d.month for d in self.dates[dec_idx]], dtype=np.int16)
        p_store_list,p_menu_list,closed_exp,in_season=[],[],[],[]
        for mo in months_dec:
            p_s,p_m=self._seasonal_probs_for_month(store_name, menu_name, int(mo))
            p_store_list.append(p_s); p_menu_list.append(p_m)
            closed=1.0 if p_m<THETA_CLOSED else 0.0
            closed_exp.append(closed); in_season.append(1.0-closed)
        dlist += [np.array(p_store_list,dtype=np.float32), np.array(p_menu_list,dtype=np.float32),
                  np.array(closed_exp,dtype=np.float32), np.array(in_season,dtype=np.float32)]
        cool_up,cool_dn=self._cooldown_from_last_spike(z); dlist += [cool_up.astype(np.float32), cool_dn.astype(np.float32)]
        enc_dow=self.cal.iloc[enc_idx]["dow"].values; enc_lin=np.expm1(enc_logy)
        dow_means={d: enc_lin[enc_dow==d].mean() if (enc_dow==d).any() else 0.0 for d in range(7)}
        same_dow=np.array([np.log1p(dow_means.get(int(d),0.0)) for d in cal_dec["dow"].values], dtype=np.float32)
        dlist += [same_dow]
        ex_dec=np.tile(ex_enc[-1] if len(self.exog_cols)>0 else np.zeros((len(self.exog_cols),),np.float32),(self.dec_len,1))
        static_dec=np.column_stack([np.full((self.dec_len,1),price,dtype=np.float32),
                                    np.tile(np.array(list(zone_oh.values()),dtype=np.float32),(self.dec_len,1))])
        base_dec=np.column_stack(dlist)
        x_dec=np.hstack([base_dec, ex_dec, static_dec])
        if USE_DOW_ONEHOT:
            dec_oh=np.eye(7,dtype=np.float32)[cal_dec["dow"].values]; x_dec=np.column_stack([x_dec, dec_oh])
        x_dec=np.nan_to_num(x_dec, copy=False)
        y_log=self.piv_logy.iloc[dec_idx, m].values.astype(np.float32)
        return {"x_enc":x_enc,"x_dec":x_dec,"y_log":y_log,
                "store_id":np.int64(self.menu_store_ids[m]),"menu_id":np.int64(self.menu_ids[m]),"menu_name":menu_name}

    @staticmethod
    def collate(batch):
        x_enc=torch.tensor(np.stack([b["x_enc"] for b in batch]),dtype=torch.float32)
        x_dec=torch.tensor(np.stack([b["x_dec"] for b in batch]),dtype=torch.float32)
        y=torch.tensor(np.stack([b["y_log"] for b in batch]),dtype=torch.float32)
        store_id=torch.tensor([b["store_id"] for b in batch],dtype=torch.long)
        menu_id =torch.tensor([b["menu_id"] for b in batch],dtype=torch.long)
        names=[b["menu_name"] for b in batch]
        return x_enc,x_dec,y,store_id,menu_id,names

class GRUSeq2SeqMonthly(nn.Module):
    def __init__(self, n_store, n_menu, enc_feat_dim, dec_feat_dim,
                 hidden=128, layers=2, dropout=0.1, horizon=7, use_attention=False, emb_dim=32):
        super().__init__()
        self.store_emb=nn.Embedding(n_store, emb_dim)
        self.menu_emb =nn.Embedding(n_menu,  emb_dim)
        self.enc=nn.GRU(input_size=enc_feat_dim+emb_dim*2, hidden_size=hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.dec=nn.GRU(input_size=1+dec_feat_dim+emb_dim*2, hidden_size=hidden, num_layers=layers, batch_first=True, dropout=dropout)
        self.head=nn.Linear(hidden,1)
    def forward(self, x_enc, x_dec, store_id, menu_id, teacher_forcing=0.0, y_log=None):
        B,T,Fe=x_enc.size(); Hh=x_dec.size(1)
        se=self.store_emb(store_id); me=self.menu_emb(menu_id)
        e_static=torch.cat([se,me],dim=-1).unsqueeze(1).repeat(1,T,1)
        enc_in=torch.cat([x_enc,e_static],dim=-1); _,h=self.enc(enc_in)
        y_prev=x_enc[:,-1:,0:1]; outs=[]
        e_static_dec=torch.cat([se,me],dim=-1).unsqueeze(1).repeat(1,Hh,1)
        for t in range(Hh):
            x_t=torch.cat([y_prev, x_dec[:,t:t+1,:], e_static_dec[:,t:t+1,:]], dim=-1)
            o,h=self.dec(x_t,h); y_hat=self.head(o); outs.append(y_hat); y_prev=y_hat
        return torch.cat(outs, dim=1)

# -----------------
# Inference helpers
# -----------------
def load_tests(test_input):
    out={}
    def _ok_name(p):
        base=os.path.basename(p); name=base.lower()
        return name.startswith("test_") and name.endswith(".csv")
    paths=sorted(glob.glob(os.path.join(test_input,"*.[cC][sS][vV]")))
    paths=[p for p in paths if _ok_name(p)]
    for p in paths:
        dt=pd.read_csv(p); dt=_normalize_columns(dt)
        out[os.path.splitext(os.path.basename(p))[0]]=dt
    if len(out)==0: raise RuntimeError("유효한 TEST_*.csv를 찾지 못했습니다.")
    return out

def load_model_from_seed(seed:int):
    save_dir=os.path.join(BASE_CKPT_DIR, f"seed{seed}")
    meta_path=os.path.join(save_dir,"model_meta.json")
    ckpt_path=os.path.join(save_dir,"best_val.pth")
    with open(meta_path,"r",encoding="utf-8") as f: meta=json.load(f)
    model=GRUSeq2SeqMonthly(n_store=len(meta["stores"]), n_menu=len(meta["menus"]),
                            enc_feat_dim=meta["enc_dim"], dec_feat_dim=meta["dec_dim"],
                            hidden=meta["hidden"], layers=meta["layers"], dropout=meta["dropout"],
                            horizon=meta["horizon"], use_attention=meta["use_attention"]).to(DEVICE)
    state=torch.load(ckpt_path, map_location=DEVICE); model.load_state_dict(state, strict=True); model.eval()
    store2id={s:i for i,s in enumerate(meta["stores"])}; menu2id={m:i for i,m in enumerate(meta["menus"])}
    pri=meta.get("priors", {}); exog_cols=meta.get("exog_cols", []); zone_keys=meta.get("zone_keys", ["z_central","z_other","z_unk"])
    return model, meta, store2id, menu2id, pri, exog_cols, zone_keys

@torch.no_grad()
def predict_one_file(models, metas_and_maps, df_t):
    grid35 = make_test_grid(df_t)
    dates = pd.to_datetime(sorted(grid35["영업일자"].unique()))
    cal = build_holiday_flags(pd.Series(dates)); cal.index = dates

    metas0, store_maps, menu_maps, pri_list, exog_cols, zone_keys = metas_and_maps

    # priors 복원
    def _restore_key(k):
        if isinstance(k,(tuple,list)): return (str(k[0]), int(k[1]))
        if isinstance(k,str) and "|" in k:
            a,b=k.split("|",1); return (a,int(b))
        return k
    pri0 = pri_list[0] if len(pri_list)>0 else {}
    smp = { _restore_key(k): float(v) for k,v in pri0.get("store_month_prob", {}).items() }
    mmp = { _restore_key(k): float(v) for k,v in pri0.get("menu_month_prob", {}).items() }
    gsp = { int(k): float(v) for k,v in pri0.get("month_global_store", {}).items() }
    gmp = { int(k): float(v) for k,v in pri0.get("month_global_menu", {}).items() }

    # test에는 exog가 없으니 zero 사용
    ds = SeqDS(grid35, enc_len=ENC_LEN, dec_len=DEC_LEN, holidays_df=cal, add_dow_oh=USE_DOW_ONEHOT,
               store_month_prob=smp, menu_month_prob=mmp, month_global_store=gsp, month_global_menu=gmp,
               exog_cols=exog_cols, price_on=False, zone_keys=zone_keys)

    assert len(ds.t_list) == 1
    enc_dim_real = ds[0]["x_enc"].shape[1]; dec_dim_real = ds[0]["x_dec"].shape[1]
    loader = DataLoader(ds, batch_size=len(ds.menu_names), shuffle=False, drop_last=False, collate_fn=SeqDS.collate)
    x_enc, x_dec, _, store_id_local, menu_id_local, names = next(iter(loader))
    x_enc = x_enc.to(DEVICE); x_dec = x_dec.to(DEVICE)

    bag = {}
    for (model, meta, store2id, menu2id, pri) in zip(models, metas0, store_maps, menu_maps, pri_list):
        if enc_dim_real != meta["enc_dim"] or dec_dim_real != meta["dec_dim"]:
            raise RuntimeError(f"Feature dim mismatch (test {enc_dim_real}/{dec_dim_real} vs meta {meta['enc_dim']}/{meta['dec_dim']}).")
        store_names = [split_store_menu(n)[0] for n in names]
        menu_train_ids = [menu2id.get(n, -1) for n in names]
        store_train_ids= [store2id.get(s, -1)  for s in store_names]
        mask = np.array([(mi>=0) and (si>=0) for mi,si in zip(menu_train_ids, store_train_ids)])
        if not mask.any(): continue
        menu_ids_t = torch.tensor([menu_train_ids[i] for i,v in enumerate(mask) if v], dtype=torch.long, device=DEVICE)
        store_ids_t= torch.tensor([store_train_ids[i] for i,v in enumerate(mask) if v], dtype=torch.long, device=DEVICE)
        yhat = model(x_enc[mask], x_dec[mask], store_ids_t, menu_ids_t, teacher_forcing=0.0, y_log=None)
        yhat = torch.expm1(yhat).clamp_min(0).squeeze(-1).cpu().numpy()
        names_kept = [n for n,keep in zip(names, mask) if keep]
        for i, mn in enumerate(names_kept):
            if mn not in bag: bag[mn] = [yhat[i].copy(), 1]
            else: bag[mn][0] += yhat[i]; bag[mn][1] += 1
    return {k: s / max(1, c) for k, (s, c) in bag.items()}

def run_ensemble_predict(seeds):
    models=[]; metas0=[]; store_maps=[]; menu_maps=[]; pris=[]; exog_cols=None; zone_keys=None
    for sd in seeds:
        m, meta, store2id, menu2id, pri, ex_cols, z_keys = load_model_from_seed(sd)
        models.append(m); metas0.append(meta); store_maps.append(store2id); menu_maps.append(menu2id); pris.append(pri)
        exog_cols = exog_cols or ex_cols
        zone_keys = zone_keys or z_keys
        print(f"[seed {sd}] loaded.")

    tests = load_tests(TEST_INPUT)
    keys = sorted(tests.keys())
    sub = pd.read_csv(SAMPLE_SUB)
    menu_cols = [c for c in sub.columns if c != "영업일자"]

    blocks = []
    for k in tqdm(keys, desc="Predict files", dynamic_ncols=True):
        df_t = tests[k]
        pred_map = predict_one_file(models, (metas0, store_maps, menu_maps, pris, exog_cols, zone_keys), df_t)
        tag = f"TEST_{k[-2:]}"; mask = sub["영업일자"].str.startswith(tag + "+")
        block = sub.loc[mask].copy()
        arr = np.zeros((DEC_LEN, len(menu_cols)), dtype=np.float32)
        for j, col in enumerate(menu_cols):
            if col in pred_map: arr[:, j] = pred_map[col]
        block.loc[:, menu_cols] = np.clip(arr, 0, None).astype(np.float32)
        blocks.append(block)

    final = pd.concat(blocks, axis=0).reset_index(drop=True)
    final.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
    print("Saved:", OUT_PATH)

# ▶️ Run inference for all seeds (independent cell)
if __name__ == "__main__":
    print("="*90); print("Run inference & build submission CSV…")
    run_ensemble_predict(SEEDS)
    print("Done.")